In [0]:
from TornAPI.Torn import User
from pyspark.sql.functions import from_unixtime
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, BooleanType, ArrayType

user_api = User(dbutils.secrets.get("Personal", "TornAPI"))

In [0]:
if spark.catalog.tableExists("torn.user.attacks"):
    current_data = spark.read.table("torn.user.attacks")
    max_date = current_data.select("started").agg({"started":"max"}).collect()
    current_max_date = max_date[0]["max(started)"]
    id_list = [id_num[0] for id_num in current_data.select("id").collect()]
else:
    current_max_date = 1681484237
    id_list = []

In [0]:
schema = StructType([
    StructField("id", IntegerType()),
    StructField("code", StringType()),
    StructField("started", IntegerType()),
    StructField("ended", IntegerType()),
    StructField("result", StringType()),
    StructField("attacker", StructType([
                StructField("id", IntegerType()),
                StructField("name", StringType()),
                StructField("level", IntegerType()),
                StructField("faction", StructType(
                    [
                        StructField("id", IntegerType()),
                        StructField("name", StringType())
                    ]))])
                ),
    StructField("defender", StructType([
            StructField("id", IntegerType()),
            StructField("name", StringType()),
            StructField("level", IntegerType()),
            StructField("faction", StructType(
                    [
                        StructField("id", IntegerType()),
                        StructField("name", StringType())
                    ])
                )
        ]
    )),
    
    StructField("respect_gain", FloatType()),
    StructField("respect_loss", FloatType()),
    StructField("chain", IntegerType()),
    StructField("is_interrupted", BooleanType()),
    StructField("is_stealthed", BooleanType()),
    StructField("is_raid", BooleanType()),
    StructField("is_ranked_war", BooleanType()),
    StructField("modifiers", StructType([
        StructField("fair_fight", FloatType()),
        StructField("war", FloatType()),
        StructField("retaliation", FloatType()),
        StructField("group", FloatType()),
        StructField("overseas", FloatType()),
        StructField("chain", FloatType()),
        StructField("warlord", FloatType())]
    ))  
])

In [0]:
attack_data = user_api.get_attacks(ts_from=current_max_date)
sp_attack_data = spark.createDataFrame(attack_data["attacks"], schema=schema)

sp_attack_data = sp_attack_data.filter(~sp_attack_data.id.isin(id_list))

sp_attack_data.write.format("delta").mode("append").saveAsTable("torn.user.attacks")